# Port-num roofline analysis (fpint_improve FFN kernel)

See `docs/hw_analysis/port_num_analysis/analysis.md` for the reuse-factor derivation.

Visualizes OI vs HP (HBM port count) and TB (TMEM bank count) for:
- large FFN GEMM (M=N=128+)
- small-M decoder (M=32, N=128+)
- small-N (M=128+, N=32)
- batched GEMV (N-padded vs batch≥32)

Uses `tools/roofline/roofline.py`.

In [ ]:
%matplotlib inline
import sys, os
REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO, 'tools', 'roofline'))
from roofline import RooflineModel
import matplotlib.pyplot as plt
import numpy as np

## 1. Architecture ceilings & kernel OI formulas

In [ ]:
# ---- HW constants ----
F_GHZ      = 0.300           # core frequency (Alveo U55C baseline)
MXU_DIM    = 32              # 32x32 systolic
PORT_WIDTH = 64              # TMEM bank / HBM AXI port = 64 B
MXU_GOPS   = 2 * MXU_DIM * MXU_DIM * F_GHZ   # 2048 * F

def hbm_gbps(HP):  return HP * PORT_WIDTH * F_GHZ
def tmem_gbps(TB): return TB * PORT_WIDTH * F_GHZ

# ---- Tile / element constants (fpint_improve FFN kernel) ----
DMA_MT, DMA_NT, DMA_KT = 128, 128, 128
MXU_KT, MXU_NT         = 32, 32
B_IN, B_W, B_Q, B_OUT  = 2.0, 0.5, 2.0, 2.0   # FP16, INT4, FP16, FP16

def tile_flops(cm, cn, ck):
    return 2 * cm * cn * ck

def hbm_bytes_per_tile(cm, cn, ck, qblk):
    """HBM -> TMEM bytes for one (mt, nt_dma, kt) DMA tile, qdir=0."""
    b_in = cm * ck * B_IN
    b_w  = ck * cn * B_W
    b_sc = (ck / qblk) * cn * B_Q
    b_zp = (ck / qblk) * cn * B_Q
    return dict(input=b_in, weight=b_w, scale=b_sc, zp=b_zp,
                total=b_in+b_w+b_sc+b_zp)

def tmem_bytes_per_block(cm, cn, ck, qblk, k_tiles):
    """Per (mt, nt_dma) block: TMEM read+write traffic summed over k_tiles."""
    nb_count = cn // MXU_NT
    kb_count = ck // MXU_KT
    hbm_fill = hbm_bytes_per_tile(cm, cn, ck, qblk)['total']
    mxu_in   = nb_count * cm * ck * B_IN            # input re-read per nb
    mxu_w    = ck * cn * B_W                        # unique weight per tile
    mxu_qp   = nb_count * kb_count * 128            # qparam per microtile
    out_wr   = cm * cn * B_OUT                      # MXU_STORE -> TMEM (once)
    out_rd   = cm * cn * B_OUT                      # DMA_STORE reads TMEM
    return k_tiles * (hbm_fill + mxu_in + mxu_w + mxu_qp) + out_wr + out_rd

def oi_hbm(cm, cn, ck, qblk, k_tiles=32):
    vols    = hbm_bytes_per_tile(cm, cn, ck, qblk)
    total_b = k_tiles * vols['total'] + cm * cn * B_OUT
    total_f = k_tiles * tile_flops(cm, cn, ck)
    return total_f / total_b

def oi_tmem(cm, cn, ck, qblk, k_tiles=32):
    total_b = tmem_bytes_per_block(cm, cn, ck, qblk, k_tiles)
    total_f = k_tiles * tile_flops(cm, cn, ck)
    return total_f / total_b

print(f'MXU peak   : {MXU_GOPS:.1f} GOps/s  (F={F_GHZ*1000:.0f} MHz)')
print('HBM BW / ridge:')
for hp in (1, 2, 4, 8):
    print(f'  HP={hp}  -> {hbm_gbps(hp):6.1f} GB/s, ridge={MXU_GOPS/hbm_gbps(hp):5.2f} FLOPs/B')
print('TMEM BW / ridge:')
for tb in (1, 2, 3, 4, 8):
    print(f'  TB={tb}  -> {tmem_gbps(tb):6.1f} GB/s, ridge={MXU_GOPS/tmem_gbps(tb):5.2f} FLOPs/B')

In [ ]:
# ---- Workload OIs for shapes we care about ----
# (cm, cn, ck) = min(M, MT), min(N, NT), min(K, KT)
workloads = [
    # label,                        cm,   cn,   ck,  qblk, k_tiles, useful_frac
    ('FFN large (M,N>=128)',         128, 128, 128, 32,  32,  1.0),
    ('small-M (M=32, N=128)',         32, 128, 128, 32,  32,  1.0),
    ('small-N (M=128, N=32)',        128,  32, 128, 32,  32,  1.0),
    ('batched GEMV B=32',             32, 128, 128, 32,  32,  1.0),
    ('batched GEMV B=64',             64, 128, 128, 32,  32,  1.0),
    ('decoder batch=1 (padded)',      32, 128, 128, 32,  32,  1/32), # only 1 of 32 rows useful
    ('GEMV N-padded (N=1->32)',      128,  32, 128, 32,  32,  1/32),
]

print(f"{'workload':<30} {'OI_HBM':>8} {'OI_TMEM':>8}  {'OI_useful_HBM':>15}")
print('-' * 68)
for name, cm, cn, ck, q, kt, frac in workloads:
    oh = oi_hbm(cm, cn, ck, q, kt)
    ot = oi_tmem(cm, cn, ck, q, kt)
    print(f'{name:<30} {oh:>8.2f} {ot:>8.2f}  {oh*frac:>15.2f}')

## 2. Baseline roofline (current arch: HP=8, TB=8)

세 대표 shape (large, small-M, small-N) 을 올려둠. `small-N` 이 HBM ridge 에 가장 근접.

In [ ]:
OI_HBM_FFN  = oi_hbm (128, 128, 128, 32)
OI_TMEM_FFN = oi_tmem(128, 128, 128, 32)
OI_HBM_SMM  = oi_hbm ( 32, 128, 128, 32)
OI_TMEM_SMM = oi_tmem( 32, 128, 128, 32)
OI_HBM_SMN  = oi_hbm (128,  32, 128, 32)
OI_TMEM_SMN = oi_tmem(128,  32, 128, 32)

def build_model(HP, TB):
    m = RooflineModel()
    m.add_compute(f'MXU 32x32 ({MXU_GOPS:.0f} GOps/s)', MXU_GOPS)
    m.add_bw(f'HBM  HP={HP}  ({hbm_gbps(HP):.1f} GB/s)',  hbm_gbps(HP))
    m.add_bw(f'TMEM TB={TB} ({tmem_gbps(TB):.1f} GB/s)', tmem_gbps(TB), linestyle='--')
    m.add_bw(f'MXU input port ({hbm_gbps(1):.1f} GB/s)',   hbm_gbps(1), linestyle=':')
    m.add_workload('FFN large @ HBM',   oi=OI_HBM_FFN,  bws=[f'HBM  HP={HP}  ({hbm_gbps(HP):.1f} GB/s)'])
    m.add_workload('FFN large @ TMEM',  oi=OI_TMEM_FFN, bws=[f'TMEM TB={TB} ({tmem_gbps(TB):.1f} GB/s)'])
    m.add_workload('small-M @ HBM',     oi=OI_HBM_SMM,  bws=[f'HBM  HP={HP}  ({hbm_gbps(HP):.1f} GB/s)'])
    m.add_workload('small-N @ HBM',     oi=OI_HBM_SMN,  bws=[f'HBM  HP={HP}  ({hbm_gbps(HP):.1f} GB/s)'])
    m.add_workload('small-N @ TMEM',    oi=OI_TMEM_SMN, bws=[f'TMEM TB={TB} ({tmem_gbps(TB):.1f} GB/s)'])
    return m

m = build_model(HP=8, TB=8)
m.plot(title=f'Baseline roofline (HP=8, TB=8, F={F_GHZ*1000:.0f} MHz)', oi_range=(0.5, 4096))
m.summary()

## 3. HP sweep (TB fixed = 8)

HBM port 수를 1→8 로 sweep 하면서 각 shape 가 compute-bound 인지 확인.

In [ ]:
HP_SWEEP = [1, 2, 4, 8]
fig, axes = plt.subplots(1, len(HP_SWEEP), figsize=(18, 5), sharey=True)
for ax, hp in zip(axes, HP_SWEEP):
    mm = RooflineModel()
    mm.add_compute(f'MXU ({MXU_GOPS:.0f})', MXU_GOPS)
    mm.add_bw(f'HBM HP={hp}',  hbm_gbps(hp))
    mm.add_bw(f'TMEM TB=8', tmem_gbps(8), linestyle='--')
    mm.add_workload('FFN large',   oi=OI_HBM_FFN, bws=[f'HBM HP={hp}'])
    mm.add_workload('small-M',     oi=OI_HBM_SMM, bws=[f'HBM HP={hp}'])
    mm.add_workload('small-N',     oi=OI_HBM_SMN, bws=[f'HBM HP={hp}'])
    mm.plot(ax=ax, oi_range=(0.5, 4096),
            title=f'HP={hp}  (ridge={MXU_GOPS/hbm_gbps(hp):.1f})',
            show_ridge=False, show_annotations=True,
            workload_fontsize=9, ceiling_fontsize=8)
plt.suptitle(f'HP sweep, TB=8 fixed  (F={F_GHZ*1000:.0f} MHz)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. TB sweep (HP fixed = 1)

TMEM bank 수를 1→8 로 sweep. small-N 은 OI_TMEM 이 낮아 TB≥3 이 필요.

In [ ]:
TB_SWEEP = [1, 2, 3, 4, 8]
fig, axes = plt.subplots(1, len(TB_SWEEP), figsize=(22, 5), sharey=True)
for ax, tb in zip(axes, TB_SWEEP):
    mm = RooflineModel()
    mm.add_compute(f'MXU ({MXU_GOPS:.0f})', MXU_GOPS)
    mm.add_bw('HBM HP=1',  hbm_gbps(1))
    mm.add_bw(f'TMEM TB={tb}', tmem_gbps(tb), linestyle='--')
    mm.add_workload('FFN @ TMEM',    oi=OI_TMEM_FFN, bws=[f'TMEM TB={tb}'])
    mm.add_workload('small-M @ TMEM', oi=OI_TMEM_SMM, bws=[f'TMEM TB={tb}'])
    mm.add_workload('small-N @ TMEM', oi=OI_TMEM_SMN, bws=[f'TMEM TB={tb}'])
    mm.plot(ax=ax, oi_range=(0.5, 4096),
            title=f'TB={tb}  (ridge={MXU_GOPS/tmem_gbps(tb):.2f})',
            show_ridge=False, show_annotations=True,
            workload_fontsize=9, ceiling_fontsize=8)
plt.suptitle(f'TB sweep, HP=1 fixed  (F={F_GHZ*1000:.0f} MHz)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. HP × TB heatmap — attainable MXU utilization [%]

HBM·TMEM bound 동시 고려. 두 panel 로 large vs small-N 비교.

In [ ]:
HP_list = [1, 2, 4, 8]
TB_list = [1, 2, 3, 4, 6, 8]

def attainable(HP, TB, oih, oit):
    return min(MXU_GOPS, hbm_gbps(HP)*oih, tmem_gbps(TB)*oit)

def heatmap_ax(ax, oih, oit, title):
    Z = np.zeros((len(HP_list), len(TB_list)))
    for i, hp in enumerate(HP_list):
        for j, tb in enumerate(TB_list):
            Z[i, j] = 100 * attainable(hp, tb, oih, oit) / MXU_GOPS
    im = ax.imshow(Z, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
    ax.set_xticks(range(len(TB_list))); ax.set_xticklabels([f'{t}' for t in TB_list])
    ax.set_yticks(range(len(HP_list))); ax.set_yticklabels([f'{h}' for h in HP_list])
    ax.set_xlabel('TB (TMEM banks)'); ax.set_ylabel('HP (HBM ports)')
    ax.set_title(f'{title}\n(OI_HBM={oih:.1f}, OI_TMEM={oit:.1f})')
    for i in range(len(HP_list)):
        for j in range(len(TB_list)):
            ax.text(j, i, f'{Z[i,j]:.0f}%', ha='center', va='center',
                    color='white' if Z[i,j] < 50 else 'black', fontweight='bold')
    return im

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
heatmap_ax(axes[0], OI_HBM_FFN, OI_TMEM_FFN, 'FFN large (M=N=128)')
heatmap_ax(axes[1], OI_HBM_SMM, OI_TMEM_SMM, 'small-M (M=32, N=128)')
im = heatmap_ax(axes[2], OI_HBM_SMN, OI_TMEM_SMN, 'small-N (M=128, N=32)')
fig.colorbar(im, ax=axes.ravel().tolist(), label='% of MXU peak', shrink=0.8)
plt.suptitle('Attainable MXU utilization vs (HP, TB)',
             fontsize=13, fontweight='bold')
plt.show()

## 6. GEMV analysis

GEMV (`y = A·x`, N=1) 시나리오:
- 알고리즘 상한 OI ≈ 4 FLOPs/B → HP=1 ridge=32 보다도 낮음 → 근본적 memory-bound
- 이 kernel 에선 `cn` min=32 이라 N-padding 시 MXU util 1/32
- Batched GEMV (B vectors stacked on M) 가 유일한 compute-bound 경로

In [ ]:
# Algorithmic GEMV OI bound (FP16 x, INT4 weight, FP16 y, large M,K)
def gemv_oi_alg(M, K):
    flops = 2 * M * K
    bytes_ = M * K * B_W + K * B_IN + M * B_OUT
    return flops / bytes_

print(f'Algorithmic GEMV OI (M=K=4096): {gemv_oi_alg(4096, 4096):.2f} FLOPs/B')
print(f'Algorithmic GEMV OI (M=K=16384): {gemv_oi_alg(16384, 16384):.2f} FLOPs/B')
print()

# This kernel's GEMV execution options
print(f"{'option':<36} {'OI_HBM':>8} {'useful':>8} {'util@HP=1':>11}")
print('-' * 68)
for name, cm, cn, frac in [
    ('(A) N-padded (N=1->32)',         128,  32, 1/32),
    ('(B) batched B=32',                32, 128, 1.0),
    ('(B) batched B=64',                64, 128, 1.0),
    ('(B) batched B>=128',             128, 128, 1.0),
    ('decoder batch=1 (M-padded)',      32, 128, 1/32),
]:
    oh = oi_hbm(cm, cn, 128, 32, k_tiles=32)
    useful = oh * frac
    bw_bound = hbm_gbps(1) * useful
    util = min(MXU_GOPS, bw_bound) / MXU_GOPS * 100
    print(f'{name:<36} {oh:>8.2f} {useful:>8.2f} {util:>10.1f}%')

# ---- Plot ----
gv = RooflineModel()
gv.add_compute(f'MXU ({MXU_GOPS:.0f})', MXU_GOPS)
gv.add_bw('HBM HP=1',  hbm_gbps(1))
gv.add_bw('HBM HP=2',  hbm_gbps(2), linestyle=':')
gv.add_bw('HBM HP=4',  hbm_gbps(4), linestyle=':')

# Use useful OI (after padding waste) so points land on true perf
gv.add_workload('GEMV alg limit',         oi=gemv_oi_alg(4096, 4096), bws=['HBM HP=1'])
gv.add_workload('N-padded (1/32 useful)', oi=oi_hbm(128, 32, 128, 32)/32, bws=['HBM HP=1'])
gv.add_workload('decoder B=1 (1/32 useful)', oi=oi_hbm(32, 128, 128, 32)/32, bws=['HBM HP=1'])
gv.add_workload('batched B=32',           oi=oi_hbm(32, 128, 128, 32),  bws=['HBM HP=1'])
gv.add_workload('batched B=64',           oi=oi_hbm(64, 128, 128, 32),  bws=['HBM HP=1'])
gv.add_workload('batched B>=128',         oi=oi_hbm(128, 128, 128, 32), bws=['HBM HP=1'])

gv.plot(title='GEMV execution modes (useful OI after padding waste)', oi_range=(0.1, 4096))
gv.summary()

## Summary

- **Large FFN (M,N≥128)** : OI_HBM≈97.5, OI_TMEM≈22.5 → HP=1, TB≥2 로 이미 compute-bound. HP=8, TB=8 은 4–8× 과잉.
- **small-M (M<128)** : weight·scale OI 는 줄지만 input OI (=128) 에서 saturation → HP=1, TB=2 로도 OK.
- **small-N (N<128)** : OI_input=N 으로 급락 → HP=2, TB=3 필요.
- **GEMV (N=1)** : 알고리즘 OI≈4, 어떤 HP/TB 로도 compute-bound 불가. 해결책은 **software batching** (B≥32).